# Taller evaluativo 03 - NumPy

Ejercicio 1 — Crear los arreglos y leer su etiqueta
Convierter los datos crudos en un arreglo y describir, como quien mira cuántas filas y columnas tiene una hoja de cálculo antes de trabajar con ella.

In [ ]:
import numpy as np

lecturas_crudas = np.array([
    [10.0, 14.5, 22.0, 30.5, 28.0, 18.0,  9.5],
    [12.0, 20.0, 35.0, 48.0, 41.0, 25.0, 14.0],
    [ 8.0, 11.0, 19.0, 27.0, 33.0, 21.0,  6.0],
    [15.0, 24.0, 31.0, 44.0, 52.0, 36.0, 17.0],
    [ 5.0,  9.0, 16.0, 22.0, 45.0, 30.0,  4.0]
])

forma = lecturas_crudas.shape
dimensiones = lecturas_crudas.ndim
total = lecturas_crudas.size
tipo = lecturas_crudas.dtype

dias = np.arange(1, 8)

pendientes = np.array([1.2, 0.9, 1.1, 1.0, 1.3])
interceptos = np.array([-0.5, 1.0, -2.0, 0.5, -1.5])

print("Forma:", forma)
print("Dimensiones:", dimensiones)
print("Total:", total)
print("Tipo:", tipo)
print("Días:", dias)
print("Pendientes:", pendientes)
print("Interceptos:", interceptos)

Forma: (5, 7)
Dimensiones: 2
Total: 35
Tipo: float64
Días: [1 2 3 4 5 6 7]
Pendientes: [1.2 0.9 1.1 1.  1.3]
Interceptos: [-0.5  1.  -2.   0.5 -1.5]


## Ejercicio 2 — Señalar celdas y recortar bloques

Se usan índices y slicing para extraer una celda, una columna, una fila y un bloque de la matriz de lecturas. Las filas representan sensores y las columnas representan días.

In [ ]:
lectura_c_dia4 = lecturas_crudas[2, 3]
ultimo_primer = lecturas_crudas[-1, 0]
dia5 = lecturas_crudas[:, 4]
semana_b = lecturas_crudas[1, :]
bloque = lecturas_crudas[1:4, 2:6]

print("Lectura de SBC-C en el día 4:", lectura_c_dia4)
print("Lectura del último sensor en el día 1:", ultimo_primer)
print("Lecturas del día 5:", dia5)
print("Lecturas de SBC-B:", semana_b)
print("Bloque SBC-B a SBC-D, días 3 a 6:")
print(bloque)
print("Forma del bloque:", bloque.shape)

Lectura de SBC-C en el día 4: 27.0
Lectura del último sensor en el día 1: 5.0
Lecturas del día 5: [28. 41. 33. 52. 45.]
Lecturas de SBC-B: [12. 20. 35. 48. 41. 25. 14.]
Bloque SBC-B a SBC-D, días 3 a 6:
[[35. 48. 41. 25.]
 [19. 27. 33. 21.]
 [31. 44. 52. 36.]]
Forma del bloque: (3, 4)


## Ejercicio 3 — Limpiar una campaña con máscaras booleanas

Se identifican las lecturas faltantes con el valor centinela -999.0 y se conservan solo las lecturas válidas, entre 0 y 500.0 inclusive. Luego se calcula el promedio de las lecturas útiles.

In [ ]:
campana_sbc_a = np.array([10.0, -999.0, 25.0, 40.0, -999.0, 8.0, 52.0, 600.0])

faltantes = campana_sbc_a == -999.0
es_util = (campana_sbc_a >= 0) & (campana_sbc_a <= 500.0)

lecturas_utiles = campana_sbc_a[es_util]
promedio_utiles = lecturas_utiles.mean()

print("Cantidad de lecturas faltantes:", int(faltantes.sum()))
print("Lecturas útiles:", lecturas_utiles)
print("Promedio de lecturas útiles:", promedio_utiles)

Cantidad de lecturas faltantes: 2
Lecturas útiles: [10. 25. 40.  8. 52.]
Promedio de lecturas útiles: 27.0


## Ejercicio 4 — Calibrar todo con operaciones vectorizadas y broadcasting

`lecturas_crudas * pendientes` produce un `ValueError` porque `lecturas_crudas` tiene forma `(5, 7)` y `pendientes` tiene forma `(5,)`. Al comparar desde la última dimensión, NumPy encuentra `7` y `5`, dimensiones incompatibles porque ninguna vale 1.

Para calibrar cada sensor con su propia recta, se convierten `pendientes` e `interceptos` en columnas de forma `(5, 1)`. Así, NumPy expande cada columna a lo largo de los siete días mediante broadcasting, sin modificar los datos originales.

In [ ]:
pendientes_col = pendientes[:, np.newaxis]
interceptos_col = interceptos[:, np.newaxis]

calibradas = lecturas_crudas * pendientes_col + interceptos_col

campana_calibrada = lecturas_utiles * 1.2 - 0.5
promedio_calibrado_sbc_a = campana_calibrada.mean()

print("Forma de calibradas:", calibradas.shape)
print("Primera fila calibrada:", calibradas[0])
print("Campaña calibrada de SBC-A:", campana_calibrada)
print("Promedio calibrado de SBC-A:", promedio_calibrado_sbc_a)

Forma de calibradas: (5, 7)
Primera fila calibrada: [11.5 16.9 25.9 36.1 33.1 21.1 10.9]
Campaña calibrada de SBC-A: [11.5 29.5 47.5  9.1 61.9]
Promedio calibrado de SBC-A: 31.9


## Ejercicio 5 — Resumir la semana con estadística por eje

Se calculan promedios generales, por sensor y por día usando `axis`. Con `axis=1` se resume cada fila o sensor; con `axis=0` se resume cada columna o día.

In [ ]:
promedio_general = calibradas.mean()
promedio_por_sensor = calibradas.mean(axis=1)
promedio_por_dia = calibradas.mean(axis=0)
maximo_por_sensor = calibradas.max(axis=1)

media_sobre_limite = calibradas[calibradas > 35.4].mean()

print("Promedio general:", np.round(promedio_general, 2))
print("Promedio por sensor:", np.round(promedio_por_sensor, 2))
print("Promedio por día:", np.round(promedio_por_dia, 2))
print("Máximo por sensor:", np.round(maximo_por_sensor, 2))
print("Media sobre el límite 35.4:", np.round(media_sobre_limite, 2))

Promedio general: 24.11
Promedio por sensor: [22.21 26.07 17.64 31.79 22.83]
Promedio por día: [10.12 16.14 25.62 35.92 42.96 27.94 10.06]
Máximo por sensor: [36.1 44.2 34.3 52.5 57. ]
Media sobre el límite 35.4: 43.28


## Ejercicio 6 — Funciones universales y reemplazo con máscara

Se calcula cuánto supera cada lectura el límite de 35.4 µg/m³, se mide la distancia absoluta de SBC-B al límite y se crea una copia con ceros en las lecturas que no superan el límite.

In [ ]:
excedente = np.maximum(calibradas - 35.4, 0)
excedente_por_sensor = np.round(excedente.sum(axis=1), 1)

desvio_sensor_b = np.round(np.abs(calibradas[1, :] - 35.4), 1)

en_alerta = calibradas.copy()
en_alerta[en_alerta <= 35.4] = 0.0

print("Excedente por sensor:", excedente_por_sensor)
print("Desviación absoluta de SBC-B:", desvio_sensor_b)
print("Celdas en alerta mayores que 0:", int((en_alerta > 0).sum()))

Excedente por sensor: [ 0.7 11.3  0.  27.3 23.7]
Desviación absoluta de SBC-B: [23.6 16.4  2.9  8.8  2.5 11.9 21.8]
Celdas en alerta mayores que 0: 8


## Ejercicio 7 — Integración: el informe de la red

Se clasifican las 35 lecturas calibradas según la escala oficial de PM2.5 y se identifica el día con mayor promedio de contaminación.

In [ ]:
buenas = int((calibradas <= 12.0).sum())
moderadas = int(((calibradas > 12.0) & (calibradas <= 35.4)).sum())
daninas_sensibles = int(((calibradas > 35.4) & (calibradas <= 55.4)).sum())
daninas_salud = int((calibradas > 55.4).sum())

dia_pico = int(dias[promedio_por_dia == promedio_por_dia.max()][0])

print(f"Lecturas por categoría: Buena={buenas}, Moderada={moderadas}, Dañina para grupos sensibles={daninas_sensibles}, Dañina para la salud={daninas_salud}")
print(f"Día más contaminado: {dia_pico} con {np.round(promedio_por_dia.max(), 1)} ug/m3")

Lecturas por categoría: Buena=9, Moderada=18, Dañina para grupos sensibles=7, Dañina para la salud=1
Día más contaminado: 5 con 43.0 ug/m3


In [ ]:
# Celda de verificación — ejecútala al final, después de todo lo demás
assert lecturas_crudas.shape == (5, 7) and lecturas_crudas.ndim == 2, "revise lecturas_crudas del Ejercicio 1"
assert forma == (5, 7) and dimensiones == 2 and total == 35 and tipo == np.float64, "revise los atributos del Ejercicio 1"
assert np.array_equal(dias, np.arange(1, 8)), "dias debe ir del 1 al 7 (el 7 incluido)"
assert np.allclose(pendientes, [1.2, 0.9, 1.1, 1.0, 1.3]) and np.allclose(interceptos, [-0.5, 1.0, -2.0, 0.5, -1.5]), "revise pendientes e interceptos"
assert lectura_c_dia4 == 27.0 and ultimo_primer == 5.0, "revise la indexación del Ejercicio 2"
assert np.array_equal(dia5, [28.0, 41.0, 33.0, 52.0, 45.0]) and semana_b.shape == (7,), "revise dia5 y semana_b del Ejercicio 2"
assert bloque.shape == (3, 4) and bloque[0, 0] == 35.0, "revise el bloque del Ejercicio 2"
assert int(faltantes.sum()) == 2 and lecturas_utiles.shape == (5,), "revise las máscaras del Ejercicio 3"
assert abs(promedio_utiles - 27.0) < 1e-9, "revise promedio_utiles del Ejercicio 3"
assert calibradas.shape == (5, 7), "calibradas debe tener forma (5, 7)"
assert np.allclose(calibradas[0], [11.5, 16.9, 25.9, 36.1, 33.1, 21.1, 10.9]), "revise la calibración del Ejercicio 4"
assert lecturas_crudas[0, 0] == 10.0, "lecturas_crudas no debe modificarse"
assert abs(promedio_calibrado_sbc_a - 31.9) < 1e-9, "revise promedio_calibrado_sbc_a del Ejercicio 4"
assert abs(promedio_general - 24.1086) < 1e-4, "revise promedio_general del Ejercicio 5"
assert np.allclose(np.round(promedio_por_sensor, 2), [22.21, 26.07, 17.64, 31.79, 22.83]), "revise promedio_por_sensor (axis=1)"
assert np.allclose(np.round(promedio_por_dia, 2), [10.12, 16.14, 25.62, 35.92, 42.96, 27.94, 10.06]), "revise promedio_por_dia (axis=0)"
assert np.allclose(maximo_por_sensor, [36.1, 44.2, 34.3, 52.5, 57.0]), "revise maximo_por_sensor"
assert abs(media_sobre_limite - 43.275) < 1e-9, "revise media_sobre_limite"
assert np.allclose(excedente_por_sensor, [0.7, 11.3, 0.0, 27.3, 23.7]), "revise excedente_por_sensor del Ejercicio 6"
assert np.allclose(desvio_sensor_b, [23.6, 16.4, 2.9, 8.8, 2.5, 11.9, 21.8]), "revise desvio_sensor_b"
assert en_alerta.shape == (5, 7) and int((en_alerta > 0).sum()) == 8, "revise en_alerta"
assert abs(calibradas[0, 3] - 36.1) < 1e-9, "calibradas no debe modificarse al crear en_alerta"
assert (buenas, moderadas, daninas_sensibles, daninas_salud) == (9, 18, 7, 1), "revise los conteos del Ejercicio 7"
assert buenas + moderadas + daninas_sensibles + daninas_salud == calibradas.size, "los cuatro conteos deben sumar 35"
assert dia_pico == 5, "revise dia_pico del Ejercicio 7"
print("Verificación completada sin errores.")

Verificación completada sin errores.
